# WPO 3: Design Reward funtion and Reward Shaping

## Old code from WPO2

Just have a look at the new Q-learning methods

The Frozen Lake environment from gymnasium. More information can be found on the Gymnasium website: https://gymnasium.farama.org/environments/toy_text/frozen_lake/.

*Description*:
Frozen lake involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. The player may not always move in the intended direction due to the slippery nature of the frozen lake.

In [ ]:
import gymnasium as gym
import numpy as np
import random
from enum import Enum
import matplotlib.pyplot as plt

In [ ]:
class Actions(Enum):
    LEFT = 0
    DOWN = 1
    RIGHT = 2
    UP = 3

def print_action(action: Actions) -> str:
    if action == Actions.LEFT:
        return '<'
    elif action == Actions.DOWN:
        return 'v'
    elif action == Actions.RIGHT:
        return '>'
    elif action == Actions.UP:
        return '^'

In [ ]:
def validate(policy, env, n_episodes: int):
    total_reward = 0
    for _ in range(n_episodes):
        total_reward += execute_episode(policy, env, False)
    print(f"Your policy has an average reward of {total_reward/n_episodes} over {n_episodes} episodes")

def execute_episode(policy, env, render):
    episode_return = 0
    terminated, truncated = False, False
    state, _ = env.reset()
    while not (terminated or truncated):
        action = policy(state)
        state, reward, terminated, truncated, _ = env.step(action.value)
        if render:
            image = env.render()
            if env.render_mode == "ansi":
                print(image)
            elif env.render_mode == "rgb_array":
                plt.imshow(image)
                plt.axis('off')
                plt.show()
        episode_return += reward
    return episode_return

In [ ]:
from typing import Dict


class QLearning:

    def __init__(self, env: gym.Env, epsilon: float):
        self.env = env
        self.eval_env = gym.make(
            'FrozenLake-v1',
            is_slippery=True,
            desc=env.unwrapped.desc,
        )
        self.epsilon = epsilon
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.qtable = np.zeros((self.env.observation_space.n, self.env.action_space.n))

    def train(self, train_episodes: int, val_frequency:float=0.05, n_eval_episodes:int=50):
        val_episodes = int(train_episodes*val_frequency)
        performance = []
        for i in range(train_episodes):
            terminated, truncated = False, False
            state, _ = self.env.reset()
            while not (terminated or truncated):
                action = self.policy(state, greedy=False)
                next_state, reward, terminated, truncated, _ = self.env.step(action.value)
                # Q-learning update
                best_next_action = np.argmax(self.qtable[next_state])
                td_target = reward + self.discount_factor * self.qtable[next_state][best_next_action]
                self.qtable[state][action.value] += self.learning_rate * (td_target - self.qtable[state][action.value]) # TD error update
                state = next_state
            if (i+1) % val_episodes == 0:
                performance += [self.evaluate(n_eval_episodes)]
        return performance + [self.evaluate(n_eval_episodes)]


    def policy(self,state: int, greedy: bool = True) -> Actions:
        if greedy or random.uniform(0, 1) > self.epsilon:
            action_idx = np.argmax(self.qtable[state])
            return Actions(action_idx)
        else:
            return Actions(random.randint(0, self.env.action_space.n - 1))
    
    def optimal_policy(self) -> Dict[int, Actions]:
        best_actions = self.qtable.argmax(axis=1)
        return {state: Actions(action) for state, action in enumerate(best_actions)}

    def print_optimal_action_map(self):
        optimal_policy = self.optimal_policy()
        action_map = [print_action(optimal_policy[state]) if self.qtable[state].any() else "." for state in range(self.env.observation_space.n)]
        row_dim = int(np.sqrt(self.env.observation_space.n))
        print("Optimal Action Map:")
        print('-'*(row_dim*4+2))
        for i in range(row_dim):
            print('| '+' | '.join(action_map[i*row_dim:(i+1)*row_dim])+' |')
            print('-'*(row_dim*4+2))
        print()

    def evaluate(self, n_episodes):
        total_reward = 0.0
        for _ in range(n_episodes):
            terminated, truncated = False, False
            state, _ = self.eval_env.reset()
            while not (terminated or truncated):
                action = self.policy(state)
                state, reward, terminated, truncated, _ = self.eval_env.step(action.value)
                total_reward += reward
        return total_reward / n_episodes

## Exercise 1

Take the Frozen Lake envirmorment from WPO2 and design a reward function to make the Q-learning agent learn faster


In [ ]:
class RewardShapingWrapper(gym.Wrapper):
    def __init__(self, env: gym.Env):
        super().__init__(env)
        self.state = None

    def reset(self, **kwargs):
        self.state, info = self.env.reset(**kwargs)
        return self.state, info

    def step(self, action):
    
        next_state, reward, terminated, truncated, info = self.env.step(action)
        
        # Modify the reward structure as needed



        self.state = next_state

        return self.state, reward, terminated, truncated, info


In [ ]:

env = gym.make(
    'FrozenLake-v1',
    is_slippery=True,
    map_name="4x4",
)
train_episodes = 1000
agent = QLearning(env, epsilon=0.7)
perf_vanilla_reward = agent.train(train_episodes=train_episodes)

env = RewardShapingWrapper(env)
agent = QLearning(env, epsilon=0.7)
perf_new_reward = agent.train(train_episodes=train_episodes)

plt.figure()
plt.plot(np.linspace(0,train_episodes,len(perf_vanilla_reward)), perf_vanilla_reward, label="Vanilla Reward")
plt.plot(np.linspace(0,train_episodes,len(perf_new_reward)), perf_new_reward, label="New Reward")
plt.legend()


## Exercise 2

Use a potetial based reward shaping 

- $R$ — original environment reward
- $\Phi: \mathcal{S}\to\mathbb{R}$ — potential function on states
- $F$ - extra reward
- $R' = R + F$ — shaped reward  

In potential-based reward shaping F has this structure, where \(s\) is the state and \(s'\) the next state
$$F(s,s') \;=\; \gamma\,\Phi(s') \;-\; \Phi(s)$$

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big( R'(s,a,s') + \gamma\max_{a'}Q(s',a') - Q(s,a)\big)$$

A good $\Phi$ speeds up learning by providing shaped intermediate rewards. **Design the potetinal function $\Phi$** (***try first with a distance***)
### Notes & caveats

- Potential-based reward shaping **does not change** the set of optimal policies (Ng et al., 1999) **iff** the shaping term is of the potential-difference form above.  
- If you use time-dependent or unbounded potentials it may break the garanties.  

### References (for further reading)

- Ng, A. Y., Harada, D., & Russell, S. (1999). _Policy invariance under reward transformations: Theory and application to reward shaping_. ICML 1999.
https://people.eecs.berkeley.edu/~pabbeel/cs287-fa09/readings/NgHaradaRussell-shaping-ICML1999.pdf

In [ ]:
class PotentialRewardShapingWrapper(gym.Wrapper):

    def __init__(self, env: gym.Env):
        super().__init__(env)
        self.state = None

    def reset(self, **kwargs):
        self.state, info = self.env.reset(**kwargs)
        return self.state, info

    def step(self, action):
        next_state, reward, terminated, truncated, info = self.env.step(action)

        
        self.state = next_state
        return next_state, reward, terminated, truncated, info

    def potential(self, state) -> float:
        
        return 0.0

Now test your wrapper and check the results. Use the various maps

In [ ]:
maps = [
[
    "SFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFF",
    "FFFFFFFG",
],
[
    "SFFFFFFF",
    "FFFFFFFF",
    "FFHHHHFF",
    "FFHHHHFF",
    "FFHHHHFF",
    "FFHHHHFF",
    "FFFFFFFF",
    "FFFFFFFG",
],
[
    "SFFFFFFF",
    "FFFFFFHF",
    "FFFFFFHF",
    "FFFFFFHF",
    "FFFFFFHF",
    "FFFFFFHF",
    "FHHHHHHF",
    "FFFFFFFG",
],
[
    "SFFFFFFF",
    "FFFFFFFF",
    "FFFHFFFF",
    "FFFFFHFF",
    "FFFHFFFF",
    "FHHFFFHF",
    "FHFFHFHF",
    "FFFHFFFG",
]
]

selected_map = maps[0] # Select the first map, try the others later
env = gym.make(
    'FrozenLake-v1',
    is_slippery=True,
    desc=selected_map,
    #render_mode="rgb_array" ## Uncomment to see the environment visually
)

agent = QLearning(env, epsilon=0.7)
perf_vanilla_reward = agent.train(train_episodes=1000)

env = PotentialRewardShapingWrapper(env)
agent = QLearning(env, epsilon=0.7)
perf_potential_reward = agent.train(train_episodes=1000)


plt.figure()
plt.plot(perf_vanilla_reward, label="Vanilla Reward")
plt.plot(perf_potential_reward, label="Potential Reward")
plt.legend()